In [9]:
import pandas as pd
import re

/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [38]:
SEC = pd.read_csv('/Users/user/Desktop/Regulatory Content Automation System/CSV/US_Securities_and_Exchange_Comission.csv')
EBA = pd.read_csv('/Users/user/Desktop/Regulatory Content Automation System/CSV/eba_articles.csv')
FRA = pd.read_csv('/Users/user/Desktop/Regulatory Content Automation System/CSV/federal_reserve_articles_with_regulator.csv')

In [39]:
SEC.columns

Index(['Title', 'Date', 'Link', 'Main Content', 'Press Release Number',
       'Release Date', 'For Immediate Release', 'Regulator Identifier'],
      dtype='object')

In [40]:
EBA.columns

Index(['Title', 'URL', 'Publication date', 'Main content',
       'Regulator identifier'],
      dtype='object')

In [41]:
FRA.columns

Index(['Article', 'Title', 'Main Content', 'Date', 'Link',
       'Regulator Identifier'],
      dtype='object')

In [42]:
import pandas as pd


sec_renamed = SEC.rename(columns={
    'Link': 'URL',
    'Release Date': 'Publication date',
    'Main Content': 'Main content',
    'Regulator Identifier': 'Regulator identifier'
})[['Title', 'URL', 'Publication date', 'Main content', 'Regulator identifier']]

eba_renamed = EBA.rename(columns={
    'Main content': 'Main content',
    'Publication date': 'Publication date',
    'Regulator identifier': 'Regulator identifier'
})[['Title', 'URL', 'Publication date', 'Main content', 'Regulator identifier']]

fra_renamed = FRA.rename(columns={
    'Date': 'Publication date',
    'Link': 'URL',
    'Main Content': 'Main content',
    'Regulator Identifier': 'Regulator identifier'
})[['Title', 'URL', 'Publication date', 'Main content', 'Regulator identifier']]


combined_df = pd.concat([sec_renamed, eba_renamed, fra_renamed], ignore_index=True)


In [43]:
combined_df

,Title,URL,Publication date,Main content,Regulator identifier
0,SEC Charges Three Texans with Defrauding Inves...,https://www.sec.gov/newsroom/press-releases/20...,"Washington D.C., April 29, 2025 —",The Securities and Exchange Commission today a...,SEC
1,"SEC Publishes New Market Data, Analysis, and V...",https://www.sec.gov/newsroom/press-releases/20...,"Washington D.C., April 28, 2025 —",The Securities and Exchange Commission’s Divis...,SEC
2,SEC Charges PGI Global Founder with $198 Milli...,https://www.sec.gov/newsroom/press-releases/20...,"Washington D.C., April 22, 2025 —",The Securities and Exchange Commission today c...,SEC
3,The EBA consults on draft amending technical s...,https://www.eba.europa.eu/publications-and-med...,30 APRIL 2025,The EBA consults on draft amending technical s...,EBA
4,The EBA issues criteria to determine when Cryp...,https://www.eba.europa.eu/publications-and-med...,25 APRIL 2025,The EBA issues criteria to determine when Cryp...,EBA
5,The EBA publishes key indicators on climate ri...,https://www.eba.europa.eu/publications-and-med...,25 APRIL 2025,The EBA publishes key indicators on climate ri...,EBA
6,Federal Reserve Board announces approval of ap...,https://www.federalreserve.gov/newsevents/pres...,4/18/2025,Federal Reserve Board announces approval of ap...,Orders on Banking Applications
7,Federal Reserve Board announces termination of...,https://www.federalreserve.gov/newsevents/pres...,3/13/2025,Federal Reserve Board announces termination of...,Enforcement Actions
8,Federal Reserve Board announces termination of...,https://www.federalreserve.gov/newsevents/pres...,4/8/2025,Federal Reserve Board announces termination of...,Enforcement Actions


In [67]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak, KeepTogether
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
import re

In [68]:

from reportlab.platypus import Image

def generate_pdf(df, pdf_filename, title, logo_path=None):
    doc = SimpleDocTemplate(pdf_filename, pagesize=letter,
                            rightMargin=40, leftMargin=40,
                            topMargin=60, bottomMargin=40)

    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name='TitleStyle', fontSize=16, leading=20, fontName='Helvetica-Bold', alignment=1, spaceAfter=10))
    styles.add(ParagraphStyle(name='Content', fontSize=11, leading=16, alignment=0))
    styles.add(ParagraphStyle(name='MainContent', fontSize=12, leading=18, alignment=1, spaceBefore=10, spaceAfter=10))
    styles.add(ParagraphStyle(name='Meta', fontSize=10, alignment=2))
    styles.add(ParagraphStyle(name='FooterLink', fontSize=10, textColor=colors.blue, alignment=1, spaceBefore=20))

    elements = []

    for idx, row in df.iterrows():
        article_elements = []

        header_data = []
        if logo_path:
            try:
                logo = Image(logo_path, width=1.2*inch, height=0.6*inch)
            except Exception as e:
                print(f"Failed to load logo: {e}")
                logo = ''
        else:
            logo = ''

        meta_text = f"<b>Article {idx+1}</b><br/><b>Date:</b> {row['Publication date']}"
        meta = Paragraph(meta_text, styles['Meta'])

        header_data.append([logo, meta])
        header_table = Table(header_data, colWidths=[2.5*inch, 3.5*inch])
        header_table.setStyle(TableStyle([
            ('VALIGN', (0, 0), (-1, -1), 'TOP'),
            ('ALIGN', (1, 0), (1, 0), 'RIGHT')
        ]))
        article_elements.append(header_table)
        article_elements.append(Spacer(1, 12))

        article_elements.append(Paragraph(row['Title'], styles['TitleStyle']))
        article_elements.append(Spacer(1, 10))

        content = row['Main content']
        if len(content) > 1000:
            sentences = re.split(r'(?<=[.!?]) +', content)
            summary = ""
            for sentence in sentences:
                if len(summary) + len(sentence) <= 1000:
                    summary += sentence + " "
                else:
                    break
            content = summary.strip()

        article_elements.append(Paragraph(content, styles['MainContent']))
        article_elements.append(Spacer(1, 10))

        article_elements.append(Paragraph(f"<b>Regulator Identifier:</b> {row['Regulator identifier']}", styles['Content']))
        article_elements.append(Spacer(1, 12))

        article_elements.append(Paragraph(f"<a href='{row['URL']}'>{row['URL']}</a>", styles['FooterLink']))

        if idx != len(df) - 1:
            article_elements.append(PageBreak())

        elements.append(KeepTogether(article_elements))

    doc.build(elements)


In [69]:
generate_pdf(
    sec_renamed,
    "/Users/user/Desktop/Regulatory Content Automation System/SEC_articles.pdf",
    "SEC Articles",
    logo_path="/Users/user/Downloads/SEC.png"
)

generate_pdf(
    eba_renamed,
    "/Users/user/Desktop/Regulatory Content Automation System/EBA_articles.pdf",
    "EBA Articles",
    logo_path="/Users/user/Downloads/EBA.png"
)

generate_pdf(
    fra_renamed,
    "/Users/user/Desktop/Regulatory Content Automation System/Federal_Reserve_articles.pdf",
    "FRA Articles",
    logo_path="/Users/user/Downloads/FRS.png"
)